# Proyecto de Regresión — E-Commerce Sales Forecast

**MP6: Proyecto de IA y Big Data — Tema 2**

**Objetivo:** Predecir las ventas diarias entre el 9 de noviembre y el 9 de diciembre de 2011.

| Conjunto | Rango |
|---|---|
| Entrenamiento + Validación | 1 dic 2010 → 8 nov 2011 |
| Test | 9 nov 2011 → 9 dic 2011 |

## 1. Carga del Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Estilo global de gráficos
sns.set_theme(style='whitegrid', palette='muted')

# Carpeta para guardar gráficos
RUTA_GRAFICOS = 'graficos/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)

df = pd.read_csv(
    'contenidoCSV/data.csv',
    encoding='latin-1'
)

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

## 2. Análisis Exploratorio de los Datos (EDA)

### 2.1 Entendimiento de los datos

In [ ]:
# 2.1.1 Dimensiones y tipos de datos
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("\nTipos de datos por columna:")
df.dtypes

In [ ]:
# 2.1.2 / 2.1.3 Primeras y últimas filas
print("--- Primeras filas ---")
display(df.head(10))
print("--- Últimas filas ---")
df.tail(10)

In [ ]:
# 2.1.4 Valores únicos por columna
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} valores únicos")

In [ ]:
# 2.1.5 Muestra de valores únicos (columnas categóricas)
columnas_categoricas = ['InvoiceNo', 'StockCode', 'Description', 'Country']
for col in columnas_categoricas:
    muestra = df[col].dropna().unique()[:10]
    print(f"\n  {col} (primeros 10 únicos):")
    print(f"  {muestra}")

In [ ]:
# 2.1.6 Estadísticas descriptivas
df.describe()

In [ ]:
# 2.1.7 Transacciones con Quantity <= 0
trans_qty_negativa = df[df['Quantity'] <= 0]
print(f"Total filas con Quantity <= 0: {len(trans_qty_negativa)} ({len(trans_qty_negativa) / len(df) * 100:.2f}%)")
trans_qty_negativa[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(10)

In [ ]:
# 2.1.8 Transacciones con UnitPrice <= 0
trans_price_negativa = df[df['UnitPrice'] <= 0]
print(f"Total filas con UnitPrice <= 0: {len(trans_price_negativa)} ({len(trans_price_negativa) / len(df) * 100:.2f}%)")
trans_price_negativa[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(10)

### 2.2 Búsqueda de valores faltantes, duplicados y erróneos

In [ ]:
# 2.2.1 Valores faltantes por columna
nulos = df.isnull().sum()
nulos_pct = nulos / len(df) * 100
resumen_nulos = pd.DataFrame({'Nulos': nulos, '% sobre total': nulos_pct.round(2)})
resumen_nulos[resumen_nulos['Nulos'] > 0]

In [ ]:
# 2.2.2 Filas sin CustomerID
sin_cliente = df[df['CustomerID'].isnull()]
print(f"Total filas sin CustomerID: {len(sin_cliente)} ({len(sin_cliente) / len(df) * 100:.2f}%)")
print("\nDistribución por país (top 10):")
sin_cliente['Country'].value_counts().head(10)

In [ ]:
# 2.2.3 Filas sin Description
sin_descripcion = df[df['Description'].isnull()]
print(f"Total filas sin Description: {len(sin_descripcion)} ({len(sin_descripcion) / len(df) * 100:.2f}%)")
sin_descripcion[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'CustomerID']].head(10)

In [ ]:
# 2.2.4 Filas sin CustomerID Y sin Description
sin_ambos = df[df['CustomerID'].isnull() & df['Description'].isnull()]
print(f"Total filas sin ambos: {len(sin_ambos)}")

In [ ]:
# 2.2.5 Duplicados exactos
duplicados = df.duplicated()
print(f"Total filas duplicadas: {duplicados.sum()} ({duplicados.sum() / len(df) * 100:.2f}%)")
df[duplicados].head(10)

In [ ]:
# 2.2.6 Formato de InvoiceDate (actualmente string)
print("Muestra de valores:")
display(df['InvoiceDate'].value_counts().head(10))
print(f"Min (lexicográfico): {df['InvoiceDate'].min()}")
print(f"Max (lexicográfico): {df['InvoiceDate'].max()}")

In [ ]:
# 2.2.7 StockCodes no estándar
stock_no_estandar = df[~df['StockCode'].str.match(r'^[0-9]{5}[A-Za-z]?$', na=False)]
print(f"Total filas con StockCode no estándar: {len(stock_no_estandar)}")
stock_no_estandar['StockCode'].value_counts().head(15)

In [ ]:
# 2.2.8 Distribución de Country
df['Country'].value_counts()

### 2.3 Búsqueda de Outliers

In [ ]:
# 2.3.1 Outliers en Quantity (IQR)
Q1_qty = df['Quantity'].quantile(0.25)
Q3_qty = df['Quantity'].quantile(0.75)
IQR_qty = Q3_qty - Q1_qty
limite_inf_qty = Q1_qty - 1.5 * IQR_qty
limite_sup_qty = Q3_qty + 1.5 * IQR_qty
outliers_qty = df[(df['Quantity'] < limite_inf_qty) | (df['Quantity'] > limite_sup_qty)]
print(f"Q1: {Q1_qty} | Q3: {Q3_qty} | IQR: {IQR_qty}")
print(f"Límite inferior: {limite_inf_qty} | Límite superior: {limite_sup_qty}")
print(f"Total outliers en Quantity: {len(outliers_qty)} ({len(outliers_qty) / len(df) * 100:.2f}%)")
print("\nTop 5 mayores:")
display(df.nlargest(5, 'Quantity')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']])
print("Top 5 menores:")
df.nsmallest(5, 'Quantity')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']]

In [ ]:
# 2.3.2 Outliers en UnitPrice (IQR)
Q1_price = df['UnitPrice'].quantile(0.25)
Q3_price = df['UnitPrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
limite_inf_price = Q1_price - 1.5 * IQR_price
limite_sup_price = Q3_price + 1.5 * IQR_price
outliers_price = df[(df['UnitPrice'] < limite_inf_price) | (df['UnitPrice'] > limite_sup_price)]
print(f"Q1: {Q1_price} | Q3: {Q3_price} | IQR: {IQR_price}")
print(f"Límite inferior: {limite_inf_price:.2f} | Límite superior: {limite_sup_price:.2f}")
print(f"Total outliers en UnitPrice: {len(outliers_price)} ({len(outliers_price) / len(df) * 100:.2f}%)")
print("\nTop 5 mayores:")
df.nlargest(5, 'UnitPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']]

In [ ]:
# 2.3.3 / 2.3.4 StockCodes y Descriptions poco frecuentes
stock_freq  = df['StockCode'].value_counts()
stock_raros = stock_freq[stock_freq <= 3]
desc_freq   = df['Description'].value_counts()
desc_raras  = desc_freq[desc_freq <= 3]
print(f"StockCodes con ≤3 apariciones: {len(stock_raros)}")
print(f"Descriptions con ≤3 apariciones: {len(desc_raras)}")

In [ ]:
# 2.3.5 / 2.3.6 Distribución por percentiles
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0]
print("Quantity:")
display(df['Quantity'].quantile(percentiles))
print("\nUnitPrice:")
df['UnitPrice'].quantile(percentiles)

### 2.4 Gráficos Auxiliares

In [ ]:
# 2.4.1 Valores faltantes por columna
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=nulos.index, y=nulos.values, ax=ax)
ax.set_title('Valores faltantes por columna', fontsize=14)
ax.set_xlabel('Columna')
ax.set_ylabel('Nº de valores nulos')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.1_valores_faltantes.png', dpi=150)
plt.show()

In [ ]:
# 2.4.2 / 2.4.3 Distribuciones de Quantity y UnitPrice
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

qty_filtrado = df[(df['Quantity'] > 0) & (df['Quantity'] <= 100)]
sns.histplot(qty_filtrado['Quantity'], bins=50, ax=axes[0], kde=True)
axes[0].set_title('Distribución de Quantity (0–100 uds.)', fontsize=13)
axes[0].set_xlabel('Quantity')

price_filtrado = df[(df['UnitPrice'] > 0) & (df['UnitPrice'] <= 20)]
sns.histplot(price_filtrado['UnitPrice'], bins=50, ax=axes[1], kde=True, color='coral')
axes[1].set_title('Distribución de UnitPrice (0–20 £)', fontsize=13)
axes[1].set_xlabel('UnitPrice (£)')

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.2_3_distribuciones.png', dpi=150)
plt.show()

In [ ]:
# 2.4.4 Top 10 países por transacciones
top_paises = df['Country'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_paises.values, y=top_paises.index, ax=ax, orient='h')
ax.set_title('Top 10 países por número de transacciones', fontsize=14)
ax.set_xlabel('Nº de transacciones')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.4_top10_paises.png', dpi=150)
plt.show()

In [ ]:
# 2.4.6 Boxplots Quantity y UnitPrice
qty_box   = df[(df['Quantity'] > 0) & (df['Quantity'] <= 100)]['Quantity']
price_box = df[(df['UnitPrice'] > 0) & (df['UnitPrice'] <= 20)]['UnitPrice']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.boxplot(y=qty_box, ax=axes[0], color='steelblue')
axes[0].set_title('Boxplot Quantity (0–100)', fontsize=13)
sns.boxplot(y=price_box, ax=axes[1], color='coral')
axes[1].set_title('Boxplot UnitPrice (0–20 £)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.6_boxplots.png', dpi=150)
plt.show()

### 2.5 Análisis Temporal

In [ ]:
# 2.5.1 Conversión de InvoiceDate a datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')
df['Fecha']     = df['InvoiceDate'].dt.normalize()
df['Mes']       = df['InvoiceDate'].dt.to_period('M')
df['DiaSemana'] = df['InvoiceDate'].dt.day_name()

print(f"Tipo resultante: {df['InvoiceDate'].dtype}")
print(f"Fecha mínima: {df['InvoiceDate'].min()}")
print(f"Fecha máxima: {df['InvoiceDate'].max()}")
print(f"Rango total:  {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} días")

In [ ]:
# 2.5.3 / 2.5.4 Transacciones por día y días sin datos
trans_por_dia  = df.groupby('Fecha').size().rename('NumTransacciones')
rango_completo = pd.date_range(start=df['Fecha'].min(), end=df['Fecha'].max(), freq='D')
dias_sin_datos = rango_completo.difference(trans_por_dia.index)

print(f"Días con datos:    {len(trans_por_dia)}")
print(f"Días sin datos:    {len(dias_sin_datos)}")
print(f"Media trans/día:   {trans_por_dia.mean():.1f}")
print(f"Máximo:            {trans_por_dia.max()} ({trans_por_dia.idxmax().date()})")
print(f"\nDías sin datos:")
print(dias_sin_datos.strftime('%Y-%m-%d').tolist())

In [ ]:
# 2.5.5 Evolución de transacciones diarias
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(trans_por_dia.index, trans_por_dia.values, linewidth=1, color='steelblue')
ax.axvline(pd.Timestamp('2011-11-25'), color='orange', linestyle='--', linewidth=1.2, label='Black Friday 2011')
ax.axvline(pd.Timestamp('2011-12-01'), color='red',    linestyle='--', linewidth=1.2, label='Diciembre (test set)')
ax.set_title('Evolución de transacciones diarias (dic 2010 – dic 2011)', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Nº de transacciones')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.5.5_transacciones_diarias.png', dpi=150)
plt.show()

In [ ]:
# 2.5.6 / 2.5.7 Transacciones por mes y por día de la semana
trans_por_mes = df.groupby('Mes').size().rename('NumTransacciones')
dias_semana   = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
trans_por_dia_semana = df.groupby('DiaSemana').size().reindex(dias_semana).rename('NumTransacciones')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

trans_por_mes.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Transacciones por mes', fontsize=13)
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x=trans_por_dia_semana.index, y=trans_por_dia_semana.values, ax=axes[1])
axes[1].set_title('Transacciones por día de la semana', fontsize=13)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.5.6_7_mes_diasemana.png', dpi=150)
plt.show()

### 2.6 Análisis de Cancelaciones (prefijo "C")

In [ ]:
# 2.6.1 / 2.6.2 Cancelaciones y cruce con Quantity
df['EsCancelacion'] = df['InvoiceNo'].str.startswith('C')
n_cancelaciones = df['EsCancelacion'].sum()

con_C_qty_neg = df[ df['EsCancelacion'] &  (df['Quantity'] < 0)]
con_C_qty_pos = df[ df['EsCancelacion'] & (df['Quantity'] >= 0)]
sin_C_qty_neg = df[~df['EsCancelacion'] &  (df['Quantity'] < 0)]
sin_C_qty_pos = df[~df['EsCancelacion'] & (df['Quantity'] >= 0)]

print(f"Total filas con prefijo 'C': {n_cancelaciones} ({n_cancelaciones / len(df) * 100:.2f}%)")
print(f"\nPrefijo 'C' + Quantity < 0  (cancelaciones normales):        {len(con_C_qty_neg):>7,}")
print(f"Prefijo 'C' + Quantity >= 0 (cancelaciones con qty positiva): {len(con_C_qty_pos):>7,}")
print(f"Sin 'C'    + Quantity < 0  (negativos huérfanos):             {len(sin_C_qty_neg):>7,}")
print(f"Sin 'C'    + Quantity >= 0 (transacciones normales):          {len(sin_C_qty_pos):>7,}")

In [ ]:
# 2.6.3 Detalle negativos huérfanos
print(f"Total negativos huérfanos: {len(sin_C_qty_neg)}")
sin_C_qty_neg[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'CustomerID']].head(10)

In [ ]:
# 2.6.5 Tasa de cancelación por mes
cancelaciones_mes = df[df['EsCancelacion']].groupby('Mes').size().rename('Cancelaciones')
total_mes         = df.groupby('Mes').size().rename('Total')
ratio_cancelacion = (cancelaciones_mes / total_mes * 100).round(2).rename('% Cancelaciones')
pd.concat([total_mes, cancelaciones_mes, ratio_cancelacion], axis=1)

In [ ]:
# 2.6.6 / 2.6.7 Gráficos de cancelaciones
datos_grafico = {
    'Normales':                  len(sin_C_qty_pos),
    'Cancelaciones (C+Qty<0)':   len(con_C_qty_neg),
    'Huérfanos (sin C+Qty<0)':   len(sin_C_qty_neg),
}
colores = ['#4CAF50', '#F44336', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wedges, _, autotexts = axes[0].pie(
    list(datos_grafico.values()), colors=colores,
    autopct=lambda p: f'{p:.1f}%\n({int(p * sum(datos_grafico.values()) / 100):,})',
    startangle=140, pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
axes[0].legend(wedges, list(datos_grafico.keys()), loc='lower center',
               bbox_to_anchor=(0.5, -0.1), ncol=1, fontsize=9)
axes[0].set_title('Tipos de transacciones', fontsize=13, pad=20)

ratio_cancelacion.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Tasa de cancelación mensual (%)', fontsize=13)
axes[1].set_xlabel('Mes')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.6_cancelaciones.png', dpi=150)
plt.show()

### 2.7 Ventas Diarias Brutas (Variable Objetivo)

In [ ]:
# 2.7.1 / 2.7.2 TotalPrice y agregación diaria
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
ventas_diarias   = df.groupby('Fecha')['TotalPrice'].sum().rename('VentasDiarias')

print(f"Venta media diaria:   £{ventas_diarias.mean():>12,.2f}")
print(f"Venta mediana diaria: £{ventas_diarias.median():>11,.2f}")
print(f"Venta máxima diaria:  £{ventas_diarias.max():>11,.2f} ({ventas_diarias.idxmax().date()})")
print(f"Venta mínima diaria:  £{ventas_diarias.min():>11,.2f} ({ventas_diarias.idxmin().date()})")
print(f"Días con ventas negativas: {(ventas_diarias < 0).sum()}")

In [ ]:
# 2.7.3 / 2.7.4 Ventas por mes y percentiles
ventas_mes = df.groupby('Mes')['TotalPrice'].sum().rename('VentasMes')
print("Ventas por mes:")
display(ventas_mes.apply(lambda x: f'£{x:,.2f}'))

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print("\nPercentiles de ventas diarias:")
ventas_diarias.quantile(percentiles).apply(lambda x: f'£{x:,.2f}')

In [ ]:
# 2.7.5 / 2.7.6 / 2.7.7 Gráficos de ventas
fig, axes = plt.subplots(3, 1, figsize=(14, 14))

# Evolución diaria
axes[0].plot(ventas_diarias.index, ventas_diarias.values, linewidth=1, color='steelblue')
axes[0].axhline(ventas_diarias.mean(), color='orange', linestyle='--', linewidth=1.2,
                label=f'Media: £{ventas_diarias.mean():,.0f}')
axes[0].axvline(pd.Timestamp('2011-11-09'), color='green', linestyle='--', linewidth=1.2,
                label='Inicio test set (9 nov)')
axes[0].set_title('Evolución de ventas diarias brutas', fontsize=13)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[0].legend()

# Por mes
ventas_mes.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Ventas totales brutas por mes', fontsize=13)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
axes[1].tick_params(axis='x', rotation=45)

# Distribución
ventas_positivas = ventas_diarias[ventas_diarias > 0]
sns.histplot(ventas_positivas, bins=40, ax=axes[2], kde=True, color='steelblue')
axes[2].axvline(ventas_positivas.mean(),   color='orange', linestyle='--', linewidth=1.5,
                label=f'Media: £{ventas_positivas.mean():,.0f}')
axes[2].axvline(ventas_positivas.median(), color='green',  linestyle='--', linewidth=1.5,
                label=f'Mediana: £{ventas_positivas.median():,.0f}')
axes[2].set_title('Distribución de ventas diarias brutas', fontsize=13)
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.7_ventas_diarias.png', dpi=150)
plt.show()

### 2.8 Top Clientes y Top Productos

In [ ]:
# 2.8.1 Top 10 clientes
ventas_cliente = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
)
top10_clientes = ventas_cliente.head(10)
total_ventas   = df['TotalPrice'].sum()

print(f"Ventas totales brutas: £{total_ventas:,.2f}")
print(f"Top 10 clientes concentran el {top10_clientes.sum() / total_ventas * 100:.1f}% de las ventas")
top10_clientes.apply(lambda x: f'£{x:,.2f}')

In [ ]:
# 2.8.2 Top 10 productos
ventas_producto = (
    df.groupby('StockCode')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
)
top10_productos = ventas_producto.head(10)
print(f"Top 10 productos concentran el {top10_productos.sum() / total_ventas * 100:.1f}% de las ventas")
top10_productos.apply(lambda x: f'£{x:,.2f}')

In [ ]:
# 2.8.3 Curva de Pareto y gráficos
ventas_cliente_pos = ventas_cliente[ventas_cliente > 0].sort_values(ascending=False)
acumulado_pct      = ventas_cliente_pos.cumsum() / ventas_cliente_pos.sum() * 100
n_clientes_80      = (acumulado_pct <= 80).sum()
print(f"Clientes que generan el 80% de las ventas: {n_clientes_80} ({n_clientes_80 / len(ventas_cliente_pos) * 100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x=[str(int(c)) for c in top10_clientes.index],
            y=top10_clientes.values, ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 clientes (£)', fontsize=12)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(x=top10_productos.index, y=top10_productos.values, ax=axes[1], color='coral')
axes[1].set_title('Top 10 productos (£)', fontsize=12)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

axes[2].plot(range(1, len(acumulado_pct) + 1), acumulado_pct.values, color='steelblue', linewidth=1.5)
axes[2].axhline(80, color='orange', linestyle='--', linewidth=1.2, label='80% ventas')
axes[2].axvline(n_clientes_80, color='red', linestyle='--', linewidth=1.2, label=f'{n_clientes_80} clientes')
axes[2].set_title('Curva de Pareto — clientes', fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.8_top_clientes_productos.png', dpi=150)
plt.show()

## 3. Limpieza de Datos

Trabajamos sobre `df_clean`, una copia del DataFrame original, para preservar `df` intacto durante todo el proceso.

In [ ]:
df_clean = df.copy()
filas_iniciales = len(df_clean)
print(f"Filas iniciales: {filas_iniciales:,}")

NameError: name 'df' is not defined

### 3.1 Eliminar filas con `Description` nula

El 100% de estas filas tienen simultáneamente `UnitPrice = 0` y `CustomerID = NaN`. No aportan información recuperable al modelo.

In [ ]:
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['Description'])
eliminadas = antes - len(df_clean)

print(f"Filas antes:      {antes:,}")
print(f"Filas eliminadas: {eliminadas:,}")
print(f"Filas después:    {len(df_clean):,}")
print(f"Description nulos restantes: {df_clean['Description'].isnull().sum()}")